# 05 - Comparative Analysis

Compare PR metrics by ecosystem/language, org type, and repo size.
Statistical tests (Kruskal-Wallis, pairwise Mann-Whitney U) quantify
whether differences are significant.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from oss_pulse.analyze.comparative import compare_ecosystems, compare_groups
from oss_pulse.visualize.comparison import plot_boxplot_comparison, plot_violin_comparison
from oss_pulse.visualize.style import PALETTE, setup_style

setup_style()

In [ ]:
# Load data and enrich with repo metadata
DATA_DIR = Path("../data/processed")
RAW_DIR = Path("../data/raw")

monthly_df = pd.read_parquet(DATA_DIR / "repo_monthly.parquet")
repos_df = pd.read_parquet(RAW_DIR / "top_repos.parquet")
pr_df = pd.read_parquet(DATA_DIR / "pr_events_featured.parquet")

# Merge language and org_type onto monthly data
monthly_enriched = monthly_df.merge(
    repos_df[["repo_name", "language", "org_type"]],
    on="repo_name",
    how="left",
)
print(f"Monthly enriched: {monthly_enriched.shape}")
print(f"Languages: {monthly_enriched['language'].nunique()}")
print(f"Org types: {monthly_enriched['org_type'].nunique()}")

In [ ]:
# Ecosystem comparison: aggregate metrics by language
# TODO: run with real data
ecosystem_stats = compare_ecosystems(monthly_enriched)
print("Ecosystem comparison:")
ecosystem_stats

In [ ]:
# Statistical test: merge rate across languages
# TODO: run with real data
merge_rate_test = compare_groups(monthly_enriched, metric="merge_rate", group_col="language")
print(f"Kruskal-Wallis statistic: {merge_rate_test['kruskal_stat']:.3f}")
print(f"Kruskal-Wallis p-value:   {merge_rate_test['kruskal_pvalue']:.4f}")
print(f"\nPairwise Mann-Whitney U (Bonferroni-corrected):")
pairwise_df = pd.DataFrame(merge_rate_test["pairwise"])
pairwise_df.sort_values("pvalue_corrected").head(10)

In [ ]:
# Visualise merge rate by language and org type
# TODO: run with real data
fig = plot_violin_comparison(
    monthly_enriched.dropna(subset=["merge_rate"]),
    metric="merge_rate",
    group="language",
    title="Merge Rate by Language",
)
fig.show()

fig = plot_boxplot_comparison(
    monthly_enriched.dropna(subset=["merge_rate"]),
    metric="merge_rate",
    group="org_type",
    title="Merge Rate by Org Type",
)
fig.show()

In [ ]:
# PR count comparison by org type
# TODO: run with real data
pr_count_test = compare_groups(monthly_enriched, metric="pr_count", group_col="org_type")
print(f"PR Count by Org Type:")
print(f"  Kruskal-Wallis p-value: {pr_count_test['kruskal_pvalue']:.4f}")

fig = plot_boxplot_comparison(
    monthly_enriched,
    metric="pr_count",
    group="org_type",
    title="Monthly PR Count by Org Type",
)
fig.show()

In [ ]:
# Merge time comparison by language
# TODO: run with real data
merge_time_test = compare_groups(
    monthly_enriched.dropna(subset=["median_merge_time_hours"]),
    metric="median_merge_time_hours",
    group_col="language",
)
print(f"Merge Time by Language:")
print(f"  Kruskal-Wallis p-value: {merge_time_test['kruskal_pvalue']:.4f}")

fig = plot_violin_comparison(
    monthly_enriched.dropna(subset=["median_merge_time_hours"]),
    metric="median_merge_time_hours",
    group="language",
    title="Median Merge Time by Language",
)
fig.show()